# Performance.py

This notebook is used to calculate the performance of a `sentence-transformers` model on a set of validation data. This validation data can either be a DIBBs TTC-generated synthetic validation set (consisting of nearly 240 thousand different enhancements across multiple LOINC name variants), or a set of data collated from a production sample (such as high confidence symedical test pairs, or medium confidence vendor test pairs).

Model performance can be assessed in two different modes: retriever, and retriever + reranker. In the first case, only the Approximate Nearest Neighbor search is performed, and the resulting search set is used to evaluate metrics. In the second case, a reranking step is applied to the results returned by the ANN search to generate a single, final prediction, which is used for metrics evaluation. The model performance is measured in several dimensions, described below. For performance tests of the ANN retriever functionality, each metric is broken down by the number of neighbors `K` retrieved by the ANN search. For tests of the retriever + reranker functionality, each metric is associated with only a single number (since the reranker re-scores and makes a single prediction out of the top-K search results):

* **Top-K accuracy**: the percentage of the time that the correct standardized code is in the K highest scoring search results. For retriever + reranker testing, this is simply **accuracy.**
* **Mean rank**: the average position (1st, 2nd, 3rd, etc.) in the list of returned neighbors (sorted by score) of the correct standardized code, when present. This metric is not computed during reranker testing.
* **Mean high similarity**: the average similarity between the nonstandard input and the _highest_ scoring search result--this result is not guaranteed to be the correct answer. For retriever testing, this similarity is the **cosine similarity**. For reranker testing, this similarity is the **cross-encoding score**, a measure of information match specifically computed by cross-encoding rerankers.
* **Mean right similarity**: the average similarity between the nonstandard input and the _correct_ standard code, if it appears in the top-K search results (for a particular search, if the right answer isn't found, than that search doesn't contribute to the mean calculation; only searches in which the right answer is present are used). As above, this similarity is cosine for retriever testing and cross-encoding score for reranker testing.
* **Mean time**: the time it takes to perform the operation. For retriever testing, this is the average time it takes to perform approximate neighbor search and return the results. For reranker testing, this is the average time it takes to re-score and re-sort the top search results, as well as to make a single prediction (note that this does not include the search time).

Additionally, the encoding time for the model (the time it takes the model to transform an input free-text string into a vector of the embedding dimension) is computed and reported once (i.e. not stratified by K-value), since this time doesn't change as K increases.

While it is possible to run this notebook on CPU, we highly recommend using an appropriately powerful GPU (e.g. the NC24 A100 series) to speed up computation. The Approximate Nearest Neighbor search we use makes searching in memory almost instantaneous, but there is still a time-cost to encode each validation string input into a vector before semantic searching. Over 60 thousand encodings, this time adds up: GPU encoding is 10-15 times faster than CPU.


## Setup

Make sure that once the compute instance is running, you activate the kernel associated with the DIBBs Env in the upper right dropdown. Its packages are correctly optimized for this notebook and avoids some `numpy` instabilities plaguing Azure.

In [ ]:
pip install azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec hnswlib

Ensure lasted version of sentence-transformers and its backend. There was a problem where some reranker models caused a "device-side" CUDA acceleration error, which blocked model instantiation. This has since been patched, with CUDA devices now directly linked to model parametricization so they launch appropriately.

In [ ]:
pip install sentence-transformers --upgrade torch --upgrade transformers --upgrade accelerate --upgrade

Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

# Authenticate to Key Vault
credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential)

# Define the workspace subscription and resources so we can instantiate a secure client
SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = 'workspaceblobstore'

# NOTE: Even though we're not directly calling any of the client functions, we do still need
# the object. Having a client instantiated acts as an authenticated connection for our compute
# instance to connect to the workspace.
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

Finally, we'll set the remainder of our imports and some global constants we'll be using.

There are four important variable blocks defined in this cell.

* **LOINC Data Variables:** These define the set of LOINC data to pull from to parse codes and their name variants. As LOINC updates roll-out or codes change, this file will change, so ensure that you're pulling from the correct, most up-to-date version.
* **MODEL VARIABLES:** These values define the files that are used to instantiate the retriever model as well as operate it efficiently. When specifying the model's name, make sure that all dashes and underscores are appropriately formatted, and make sure that each text section in the model's file path appears in the model's name variable here (e.g. the ending tag `_no_cn` occurs on the final TTC model, denoting the exlcusion of consumer name from the model's coverage). The DIBBs TTC model has an embedding dimension of 1024, but if you use this notebook to test other models, their embedding size can be found on the respective Hugging Face model cards (the two most common values for any BERT-based transformer are 1024 and 768). When specifying the model's embedding file, note any prefixing directories in the filepath and make sure those are included. The TTC team's final model embedding file lives in the subdirectory `embeddings/final/fine_tuned`, for example, and leaving out any of those folders will cause an operation later to fail. The model's Index variable should auto-populate based on the model's name.
* **RERANKER VARIABLES:** The only variable that affects the reranker is its name (since it has no embedding or index file). As with specifying model name, ensure everything is correctly formatted.
* **VALIDATION VARIABLES:** The main variable that affects validation is the name of the file that the test set should be loaded from. There are three options for this file, depending on whether you wish to test model performance against synthetic test data, "high confidence" symedical data, or "medium confidence" vendor-sampled data. Those file names are, respectively: `./prod_emulated_validation_set_no_cn.txt`, `symedical_high_confidence_test_pairs.txt`, and `medium_confidence_vendor_test_pairs.txt`. If you are testing with one of the latter two files, the variable `MAP_TO_NUMERIC_CODES` will be set to True, since these pairs parsed from the APHL sample were expressed as numeric codes rather than code strings, and this will tell the model to make one additional translation step.

In almost every case, the JSONL variables at the bottom of the block can be ignored. They are for one specific JSONL parsing task.


In [ ]:
import os
import random
import time
from typing import List

# LOINC DATA VARIABLES
SNOINC_CODES_FILE = "./loinc_lab_names_20260223.csv"
DATE = SNOINC_CODES_FILE.split("_")[-1].split(".")[0]

# MODEL VARIABLES
MODEL_NAME = "intfloat_e5-large-v2_0.3_1e05_mnrl_tuned_450000_1e06_no_cn"
EMBEDDING_SIZE = 1024
EMBEDDING_FILE = f"embeddings/final/fine_tuned/loinc_lab_names_{MODEL_NAME.replace('/', '_')}_{DATE}"
INDEX_FP = f"hnswlib_index_{MODEL_NAME.replace('/', '_')}.index"

# RERANKER VARIABLES
RERANKER_NAME = "sentence-transformers/gtr-t5-large"

# VALIDATION VARIABLES
VALIDATION_FILE = "./prod_emulated_validation_set.txt"
MAP_TO_NUMERIC_CODES = "confidence" in VALIDATION_FILE


# These variables are NOT necessary for regular use--this is purely for
# JSONL validation. Just leave this first boolean set to False unless
# you want to verify the embedding integrity of JSONL (which is a 
# very slooooow operation).
USE_JSONL_EMBEDDINGS = False
MODEL_SPECIFIC_JSONL_DIR = f"loinc_lab_names_{MODEL_NAME.replace('/', '_')}_{DATE}"
PATH_TO_JSONL_DIR = f"embeddings/refined/split/{MODEL_SPECIFIC_JSONL_DIR}"

The TTC team has found that Approximate Nearest Neighbor search using a GPU-backed compute instance is the most performant evaluation option. ANN is roughly twice as fast as Exact search, even with GPU-boosting, due to the speed of retrieval. For this reason, we enforce that a GPU must be available.

In [ ]:
import torch
assert torch.cuda.is_available()

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC storage container. Any `.txt` or `.csv` files just need to be uploaded to the desired directory within the DIBBs storage container. Unlike with more complex file types, there is no need to manually turn these files into Azure Data Assets before using them. They can simply be directly loaded from storage once they're uploaded.

In [ ]:
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)

## Step 2: Unpickle Embeddings

Using our mounted file system, we can directly open the embedding file and unpickle it. Remember, each embedding file is stored as a dictionary of not just the embeddings computed by the `sentence-transformers` model, but the standard LOINC codes associated with those embeddings. These are important later for measuring accuracy.

In [ ]:
import pickle
import json
import numpy as np
import os

if not USE_JSONL_EMBEDDINGS:
    # Load up the pre-computed embeddings from the datastore
    with fs.open(EMBEDDING_FILE) as fp:
        cache_data = pickle.load(fp)

    name_codes = cache_data["codes"]
    embeddings = cache_data["embeddings"]
    embeddings = embeddings.cpu().numpy()

    # TTC model includes four name variants but not consumer name, so there
    # are just over 335k vectors that are eligible
    print(f"{len(name_codes)} embeddings loaded")
    assert len(name_codes) > 335_000

else:
    # Construct a list of all JSONL files we'll need to open
    EMBEDDING_FILE_LIST = [
        os.path.join(PATH_TO_JSONL_DIR, os.path.basename(p))
        for p in fs.ls(PATH_TO_JSONL_DIR)
        if p.endswith(".jsonl")
    ]

    cache_data = []
    for file in EMBEDDING_FILE_LIST:
        with fs.open(file) as f:
            cache_data.extend(
                json.loads(line)
                for line in f
                if line.strip()
            )

    name_codes = [item["description"] for item in cache_data]
    embeddings = [np.array(item["descriptionVector"], dtype=np.float32)
                for item in cache_data]
    loinc_type = [item["type"] for item in cache_data]

    # embeddings: list of 1D vectors -> stack to 2D array for hnswlib
    embeddings = np.stack(embeddings, axis=0)  # shape (N, D)


## Step 3: Load HNSW Index

Whether computed from a previous Azure run, or computed locally and uploaded, we will use the embeddings to populate an HNSW index for fast Approximate Nearest Neighbor searching. The parameter values below govern the depth / connectivity of the search, but note that if the index was previously constructed, only the `EF_SEARCH` value will impact performance.

The `hnswlib` package _cannot_ directly open Azure binary files, which is how the FileSystemMount accesses and passes them. So what we need to do instead is first copy the file from the remote mount to local, working memory, and then we can access and open it. Once we've done that, it should be locally persisted for the remainder of our session.

During operation, this cell will create a temporary copy of the `.index` file in local, working memory. At the end of the cell, the file will be remote copied to Blob Storage and then deleted from local memory (on subsequent runs, it will simply be fetched from remote storage). You can verify the file has been cleaned by checking the sidebar to the left, under `Notebooks`.


In [ ]:
import hnswlib

# MODEL DIRECTORY
# IMPORTANT: Make sure this sub-folder is set to the correct location
# where the model's respective index file lives (or should live). For
# fine-tuned models, this should be "fine_tuned/". For TSDAE models that
# haven't been tuned, this should be "tsdae/". For all other untrained
# models, it should be "".
MODEL_SUB_DIR = "final/"
# MODEL_SUB_DIR = "tsdae/"

# ANN INDEX VARIABLES
EF_CONSTRUCTION = 400
M_VALUE = 64
EF_SEARCH = 400

# Load up or create an index over the embedding data
index = hnswlib.Index(space="cosine", dim=EMBEDDING_SIZE)

# Azure will check blob storage first using the file mount
print("Checking for cached ANN index...")
if fs.exists("indexes/" + MODEL_SUB_DIR + INDEX_FP):
    print("  Found cached index. Loading it...")

    # First, try to regularly load the index, in case we copied it here
    # from a previous run
    try:
        index.load_index(INDEX_FP)
    
    # If we can't open the file (because it's AzureML binary), then we
    # can create a local ported copy and open from that
    except:
        try:
            fs.get("indexes/" + MODEL_SUB_DIR + INDEX_FP, '.')
            index.load_index(INDEX_FP)
        
        # If that doesn't work then the file is beyond the reach of mortal
        # hands and is best left undisturbed, like all sleeping gods
        except:
            print("Could not copy or load index")
    
else:
    print("No locally cached index found. Creating hierarchical index...")
    index.init_index(
        max_elements=len(embeddings), ef_construction=EF_CONSTRUCTION, M=M_VALUE
    )
    index.add_items(embeddings, list(range(len(embeddings))))

    # Default is to save to local, working memory, so we'll need to remote copy
    # to Azure blob storage just like the reverse of copying from blob storage
    # Also clean up the local copy to avoid surplus memory charges
    index.save_index(INDEX_FP)
    fs.put(INDEX_FP, "/indexes/" + MODEL_SUB_DIR)
os.remove(INDEX_FP)

# The index should be holding approximately 276k embeddings so it better exceed 0
assert index.get_current_count() > 0
index.set_ef(EF_SEARCH)

## Step 4: Load Validation Set

With our file system mount, loading the validation set and preparing it for evaluation is straightforward. No need to worry about local copying for this data, Azure's `fs.open()` can simply parse the binary into a string codec for us.

In [ ]:
print("Loading validation set...")


examples = []
with fs.open(VALIDATION_FILE) as fp:
    for line in fp:
        # Blob storage is bytes-based, so we need to decode before string operations
        line_str = line.decode("utf-8")
        if line_str.strip() != "":
            examples.append(line_str.strip().split("|"))

# Make sure we loaded samples from the validation file
assert len(examples) > 0

if MAP_TO_NUMERIC_CODES:
    # Unfortunately, some of the symedical data is actually duplicated queries
    # that map to the same things--they can be wholly removed without
    # affecting representativeness
    examples = list(set([(x[0], x[1]) for x in examples]))
    print(len(examples))

    # With the duplicates corrected, we need to build up the dictionary mapping
    # that will let us map symedical numeric codes to scored strings
    loinc_names_to_codes = {}

    with fs.open(SNOINC_CODES_FILE) as fp:
        # First line is a header giving the column names
        lines_seen = 0
        for line in fp:
            if lines_seen == 0:
                lines_seen += 1
                continue
            
            # Azure Blob Storage is bytes-based, so we need to apply utf decoding
            # before we can use string operations
            line_str = line.decode("utf-8")
            if line_str.strip() != "":
                names = line_str.strip().split("|")
                # Skip lines that aren't real entries (formatting artifacts)
                if len(names) >= 4:
                    # Short
                    if names[8].strip() != "":
                        loinc_names_to_codes[names[8].strip()] = names[0].strip()
                    # Long
                    if names[9].strip() != "":
                        loinc_names_to_codes[names[9].strip()] = names[0].strip()
                    # Display
                    if names[10].strip() != "":
                        loinc_names_to_codes[names[10].strip()] = names[0].strip()
                    # Fully Specified
                    if names[13].strip() != "":
                        loinc_names_to_codes[names[13].strip()] = names[0].strip()
                    # Consumer
                    if names[14].strip() != "":
                        loinc_names_to_codes[names[14].strip()] = names[0].strip()
    
print(f"{len(examples)} validation samples loaded.")

## Step 5: Instantiate Models

Before running the performance tests, we need to instantiate all models we're using in the pipeline (either just the retriever, or the retriever and the reranker). The cell below checks whether the models exist in local, working memory, and if they don't, fetches them from remote storage.

While we offer a setting to load a `Transformer` model as a reranker, we do not recommend this approach technically or architecturally. See the Fine Tuning notebook for more details, but in most use cases, the `RERANKER_IS_A_TRANSFORMER` parameter should remain `False`.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

FETCH_TRAINED_MODEL = True
# The Azure directory where the retriever lives. Should be either
# `fine_tuned/` or `tsdae/`.
TRAINED_DIR = "fine_tuned/"

FETCH_RERANKER = True
RERANKER_DIR = "rerankers/"
RERANKER_IS_A_TRANSFORMER = False

if FETCH_TRAINED_MODEL:
    # First, check if the model exists locally--if it does, nothing to do here
    if os.path.exists(MODEL_NAME):
        print("Model exists locally, loading it...")

    # If there isn't a local copy, we'll fetch it from remote
    else:   
        if fs.exists("models/" + TRAINED_DIR + MODEL_NAME):
            print("Found trained model, loading from remote...")
            fs.get("models/" + TRAINED_DIR + MODEL_NAME, '.')
            print("Model loaded to local memory.")
        else:
            print("Could not find model at specified path.")
            print(
                "Check model name (esp. parameter numbers, underscores, and dashes) and fetch directory."
            )
    
print("Instantiating language model...")
model = SentenceTransformer(MODEL_NAME)

if FETCH_RERANKER:
    if os.path.exists(RERANKER_NAME):
        print("Reranker exists locally, loading it...")

    # If there isn't a local copy, we'll fetch it from remote
    else:   
        if fs.exists("models/" + RERANKER_DIR + RERANKER_NAME):
            print("Found trained reranker, loading from remote...")
            fs.get("models/" + RERANKER_DIR + RERANKER_NAME, '.')
            print("Reranker loaded to local memory.")
        else:
            print("Could not find reranker at specified path.")
            print(
                "Check reranker name (esp. parameter numbers, underscores, and dashes) and fetch directory."
            )
    
    if RERANKER_IS_A_TRANSFORMER:
        print("Instantiating transformer-based reranker...")
        reranker = SentenceTransformer(RERANKER_NAME)
    else:
        print("Instantiating cross-encoding reranker...")
        reranker = CrossEncoder(RERANKER_NAME)

## Step 6: Run Performance Tests

This cell carries out the metrics test with the model and scores its performance on the validation data. It's largely a dictionary-based tracking function that accumulates some numbers into lists partitioned out by the K-value associated with the run.

When we use the `hnswlib` API to perform ANN, we get a pretty nested structure of a pair of lists denoting the search results and the _distances_ of those results to the input query. The only nuance to this function is unpacking those lists, converting distances into scores (since we want to measure similarity), and pairing up the found neighbor result with the standard LOINC code it represents, using our earlier unpickled Corpus ID indices.

This notebook offers the capacity to run performance evaluation in two primary ways.

The first evaluation method is **Multi-K**, which is a method in which multiple values of `K` are tested in the array of `K_VALUES`. This method is designed to test broad model performance, such as establishing benchmark accuracy and similarity at each K-Value. When first evaluating a model (e.g. when testing a model that hasn't been domain adapted using either TSDAE or fine-tuning), we recommend this approach to determine whether the model is worth pursuing further. To use this mode, set the following parameters as indicated:

* `SHOW_MULTI_K_STATS`: set this to True
* `USE_RERANKER`: set this to False
* `K_VALUES`: include several numbers in the array to define the K-neighbors retrieved; we recommend 1, 3, 5, and 10

The second evaluation method is **With Reranker**, which is a method in which only a single value of K-neighbors is tested (typically 10 or 25), but a secondary language model applies an additional round of scoring on those K retrieved candidates to determine which is the single best fit for the input query. A reranker is an LLM much like the retriever that performs nearest neighbor search, but rather than compare strings based on vector distance, it uses a much more expensive _cross encoding_ score to evaluate semantic information relevance. This is why we perform a first pass using the cheap and efficient retriever before a more thorough, expensive comparison using the reranker. Once a model has its benchmarks established using the above evaluation method, this method is suitable for testing how that model might perform in production, where only a single code can be returned for a given input. To use this mode, set the following parameters as indicated:

* `SHOW_MULTI_K_STATS`: set this to False
* `USE_RERANKER`: set this to True
* `K_VALUES`: include in the array only a single number of neighbors for the first pass retriever to fetch; we recommend 10 or 25. Importantly, this value should still be an array, e.g. `K_VALUES = [10]`.


In [ ]:
from sentence_transformers import util
import json

# IMPORTANT: The default set of K-Values should be [1,3,5,10].
# HOWEVER, if you're testing the reranker, then this should be a list
# of only a single element: K_VALUES = [10].
K_VALUES = [10]

# Set to True if you want to see stats computed for K-Values. For reranker
# operations, this should be set to False.
SHOW_MULTI_K_STATS = False
if SHOW_MULTI_K_STATS:
    highest_cosine_sims = {k: [] for k in K_VALUES}
    right_cosine_sims = {k: [] for k in K_VALUES}
    ranks = {k: [] for k in K_VALUES}
    examples_with_correct_output_in_top_k = {k: 0.0 for k in K_VALUES}
    num_examples_not_terminated = {k: len(examples) for k in K_VALUES}

USE_RERANKER = True
if USE_RERANKER:
    # Stats for Re-Ranker Testing
    reranker_accuracy = 0.0
    num_examples_not_terminated = len(examples)
    reranker_high_sims = []
    reranker_right_sims = []
    ranking_times = []

# Optionally, you can collect the distribution of search results and reranker
# scores for each query in the validation file during testing. This can be 
# useful for examining cutoff heuristics and confidence thresholds later.
COLLECT_SCORING_DISTRIBUTION = False
SCORING_DISTRIBUTION_OUT_FILE = "scored_search_results.json"
if COLLECT_SCORING_DISTRIBUTION:
    scoring_dist = []

print("Predicting and computing stats for validation set...")

encoding_times = []
search_times = {k: [] for k in K_VALUES}

# Always randomize before evaluation to avoid cold-initialization bias
random.shuffle(examples)

for i, e in enumerate(examples):

    # We're impatient people, this helps keep us sane by seeing progress
    # is happening
    if i % 10_000 == 0:
      print(f"Calculated {i} of {len(examples)} examples.")

    correct_code = e[0].strip()
    nonstandard_in = e[1].strip()

    # Depending on whether we're using ANN, the encoded non-standard input
    # must be kept in CPU (ANN) or GPU (exact search)
    start = time.time()
    enc = model.encode(nonstandard_in)
    encoding_times.append(time.time() - start)

    for k in K_VALUES:
        if COLLECT_SCORING_DISTRIBUTION:
            code_search = {
                "nonstandard_in": nonstandard_in,
                "correct_code": correct_code
            }
        start = time.time()

        # Note that ANN works using _distances, so we have to convert
        # to scores (cosine distance is unit-normalized to always be 
        # length-1, so we can just subtract 1 - dist)
        embedding_ids, distances = index.knn_query(enc, k=k)
        hits = [
            {"corpus_id": id, "score": 1 - dist}
            for id, dist in zip(embedding_ids[0], distances[0])
        ]
        hits = sorted(hits, key=lambda x: x["score"], reverse=True)

        search_times[k].append(time.time() - start)
        
        # Early termination if we have multiple 1.0 similarities (indicates CN 
        # one-to-many overload), there are 3 or more results with the same
        # code string (indicates impossible to differentiate CN) at the front
        # of the search results, or 5 or more results with the same code string
        # anywhere in the results
        num_perfects = len([x for x in hits if x["score"] >= 1])
        leading_code = name_codes[hits[0]["corpus_id"]]
        leading_copies = 0
        max_copies = 0
        unique_keys = {}
        for i, h in enumerate(hits):
            mapped_sentence = name_codes[h["corpus_id"]]
            if not mapped_sentence in unique_keys:
                unique_keys[mapped_sentence] = 0
            if mapped_sentence == leading_code:
                leading_copies += 1
            new_count = unique_keys[mapped_sentence] + 1
            if new_count > max_copies:
                max_copies = new_count
            unique_keys[mapped_sentence] = new_count
        
        if num_perfects > 1 or leading_copies >= 3 or max_copies >= 5:
            if SHOW_MULTI_K_STATS:
                num_examples_not_terminated[k] = num_examples_not_terminated[k] - 1
            elif USE_RERANKER:
                num_examples_not_terminated -= 1
            continue
        
        if SHOW_MULTI_K_STATS:
            highest_cosine_sims[k].append(hits[0]["score"])

        if COLLECT_SCORING_DISTRIBUTION:
            search_results = []
            search_scores = []
            for h in hits:
                search_results.append(name_codes[h["corpus_id"]])
                search_scores.append(float(h["score"]))
            code_search["search_results"] = search_results
            code_search["search_scores"] = search_scores

        if USE_RERANKER:
            # We need to reconstruct each code possibility before feeding it
            # into the reranker
            query_hits = []
            for h in hits:
                mapped_sentence = name_codes[h["corpus_id"]]  # ty: ignore
                query_hits.append(mapped_sentence)

            if RERANKER_IS_A_TRANSFORMER:
                start = time.time()
                encoded_neighborhood = reranker.encode(query_hits)
                similarities = reranker.similarity(enc, encoded_neighborhood)
                ranking_times.append(time.time() - start)
                max_idx = np.argmax(similarities)
                
                predicted_code = query_hits[max_idx]
                predicted_score = similarities[0, max_idx]

            else:
                # Now we can cross-encode to get the scores for all pairs of codes
                start = time.time()
                ranks = reranker.rank(nonstandard_in, query_hits)
                ranking_times.append(time.time() - start)
                hits_idx = ranks[0]['corpus_id']

                predicted_code = query_hits[hits_idx]
                if MAP_TO_NUMERIC_CODES:
                    predicted_code = loinc_names_to_codes[predicted_code]
                predicted_score = ranks[0]['score']

                # Reranker score order will be different than retriever order, so
                # we need to store this list too if we're distribution collecting
                if COLLECT_SCORING_DISTRIBUTION:
                    reranker_results = []
                    for r in ranks:
                        reconstructed_code = query_hits[r["corpus_id"]]
                        reranker_score = r["score"]
                        reranker_results.append( [reconstructed_code, reranker_score] )
                    code_search["reranker_results"] = reranker_results
                    scoring_dist.append(code_search)

            # Save the stats output
            if predicted_code == correct_code:
                reranker_accuracy += 1.0
                reranker_right_sims.append(predicted_score)
            reranker_high_sims.append(predicted_score)

        else:
            # Check if correct answer is in the returned search results
            correct_in_top_k = False
            for idx, h in enumerate(hits):
                mapped_sentence = name_codes[h["corpus_id"]]  # ty: ignore
                # If we need to take things to the level of the numeric code,
                # one more mapping to do
                if MAP_TO_NUMERIC_CODES:
                    mapped_sentence = loinc_names_to_codes[mapped_sentence]
                if mapped_sentence == correct_code:
                    correct_in_top_k = True
                    # Hits is a 0-indexed list, so translate the index to the nth
                    # element of the list
                    ranks[k].append(idx+1)
                    right_cosine_sims[k].append(hits[idx]["score"])
                    break
            if correct_in_top_k:
                examples_with_correct_output_in_top_k[k] += 1.0
        

if COLLECT_SCORING_DISTRIBUTION:
    json_write_obj = {
        "query_score_distribution": scoring_dist
    }
    with open(SCORING_DISTRIBUTION_OUT_FILE, 'w') as fp:
        json.dump(json_write_obj, fp)

if SHOW_MULTI_K_STATS:
    mean_encoding_time = round(float(sum(encoding_times)) / float(len(encoding_times)), 3)
    print(f"  Mean Encoding Time: {mean_encoding_time} seconds")

    for k in K_VALUES:
        if len(highest_cosine_sims[k]) > 0:
            mean_high_cosine_sim = round(float(sum(highest_cosine_sims[k])) / float(len(highest_cosine_sims[k])), 3)
        else:
            mean_high_cosine_sim = "n/a"
        if len(right_cosine_sims[k]) > 0:
            mean_right_cosine_sim = round(float(sum(right_cosine_sims[k])) / float(len(right_cosine_sims[k])), 3)
        else:
            mean_right_cosine_sim = "n/a"
        mean_encoding_search_time = round(float(sum(search_times[k])) / float(len(search_times[k])), 3)
        top_k_accuracy = round(examples_with_correct_output_in_top_k[k] / float(num_examples_not_terminated[k]), 5)
        if (len(ranks[k])) > 0:
            mean_rank = round(float(sum(ranks[k])) / float(len(ranks[k])), 3)
        else:
            mean_rank = "n/a"

        print(f"  Trial: Value for Top-K at K = {k}")
        print(f"   Number of queries considered: {num_examples_not_terminated[k]}")
        print(f"    Top-K Accuracy (Excluding Impossible Decisions): {top_k_accuracy * 100.0}%")
        print(f"    Mean Rank of Correct Code (when present): {mean_rank}")
        print(f"    Mean Highest Cosine Similarity: {mean_high_cosine_sim}")
        print(f"    Mean Correct Cosine Similarity: {mean_right_cosine_sim}")
        print(f"    Mean Search Time: {mean_encoding_search_time}")

elif USE_RERANKER:
    reranker_accuracy = round(reranker_accuracy / float(num_examples_not_terminated), 5)
    # reranker_accuracy = round(reranker_accuracy / 100.0, 5)
    if len(reranker_high_sims) > 0:
        mean_reranker_high_sim = round(float(sum(reranker_high_sims)) / float(len(reranker_high_sims)), 3)
    else:
        mean_reranker_high_sim = "n/a"
    if len(reranker_right_sims) > 0:
        mean_reranker_right_sim = round(float(sum(reranker_right_sims)) / float(len(reranker_right_sims)), 3)
    else:
        mean_reranker_right_sim = "n/a"
    if len(ranking_times) > 0:
        mean_reranker_time = round(float(sum(ranking_times)) / float(len(ranking_times)), 3)
    else:
        mean_reranker_time = "n/a"
    if len(search_times[K_VALUES[0]]) > 0:
        mean_encoding_search_time = round(float(sum(search_times[K_VALUES[0]])) / float(len(search_times[K_VALUES[0]])), 3)
    else:
        mean_encoding_search_time = "n/a"

    print(f"Re-Ranker Accuracy: {reranker_accuracy * 100.0}%")
    print(f"Number of queries not terminated early: {num_examples_not_terminated}")
    print(f"Mean Highest Reranker Score: {mean_reranker_high_sim}")
    print(f"Mean Correct Reranker Score: {mean_reranker_right_sim}")
    print(f"Mean search time at K = {K_VALUES[0]}: {mean_encoding_search_time}")
    print(f"Mean Re-Ranking Time: {mean_reranker_time}")
